# LeetCode 1190 — Reverse Substrings Between Each Pair of Parentheses bằng Python

Link đề: https://leetcode.com/problems/reverse-substrings-between-each-pair-of-parentheses/ · Độ khó: **Medium** · Chủ đề: String, Stack, Bracket Sequences

Skeleton LeetCode:

```python
class Solution:
    def reverseParentheses(self, s: str) -> str:
        ...
```

Ghi chú: xử lý **ngoặc lồng nhau** là "đất diễn" của **stack** (ngăn xếp). Bài này có ba mức lời giải: mô phỏng trực tiếp `O(n^2)`, stack các đoạn chuỗi, và mẹo **"đường hầm" (wormhole / teleport)** đi xuyên qua cặp ngoặc rồi đổi chiều, đạt `O(n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đảo ngược phần chuỗi trong mỗi cặp ngoặc, **từ trong ra ngoài**, và xóa hết ngoặc.
2. Dùng **stack** để ghép cặp ngoặc mở/đóng và để lưu "ngữ cảnh" bên ngoài khi đi vào ngoặc mới.
3. Phân tích vì sao cách đảo trực tiếp tốn `O(n^2)` (một ký tự có thể bị đảo nhiều lần).
4. Nắm mẹo `O(n)`: tiền xử lý **cặp ngoặc tương ứng** `pair[i]`, rồi đi qua chuỗi, gặp ngoặc thì **nhảy sang ngoặc đối diện và đổi chiều**.
5. Tránh lỗi: đảo theo thứ tự sai (ngoài trước trong), quên xóa ngoặc, dùng `str` cộng dồn trong vòng lặp gây chậm.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ thường tiếng Anh và các dấu ngoặc tròn `(`, `)`. Hãy **đảo ngược** chuỗi con nằm trong mỗi cặp ngoặc khớp nhau, bắt đầu từ cặp **trong cùng**. Kết quả **không chứa ngoặc**.

**Ví dụ 1:** `s = "(abcd)"` → `"dcba"`.

**Ví dụ 2:** `s = "(u(love)i)"` → `"iloveu"`. Đảo `"love"` thành `"evol"` trước → `"(uevoli)"` → đảo cả cụm → `"iloveu"`.

**Ví dụ 3:** `s = "(ed(et(oc))el)"` → `"leetcode"`. Đảo `"oc"` → `"co"`, rồi `"etco"` → `"octe"`, cuối cùng đảo toàn bộ.

**Ràng buộc:** `1 <= s.length <= 2000`; `s` chỉ gồm chữ thường và ngoặc; ngoặc luôn **cân bằng**.

In [1]:
import random
import time
from typing import List

examples = [
    ("(abcd)", "dcba"),
    ("(u(love)i)", "iloveu"),
    ("(ed(et(oc))el)", "leetcode"),
    ("a(bcdefghijkl(mno)p)q", "apmnolkjihgfedcbq"),
]
for s, want in examples:
    print(f"{s!r:28} -> {want!r}")

'(abcd)'                     -> 'dcba'
'(u(love)i)'                 -> 'iloveu'
'(ed(et(oc))el)'             -> 'leetcode'
'a(bcdefghijkl(mno)p)q'      -> 'apmnolkjihgfedcbq'


## 3. Kiến thức nền

### 3.1 Đảo chuỗi / list trong Python

- `s[::-1]` tạo chuỗi đảo ngược (slice bước `-1`).
- `lst.reverse()` đảo list **tại chỗ**; `lst[l:r] = lst[l:r][::-1]` đảo một đoạn.
- `str` là **bất biến (immutable)**: mọi "sửa" đều tạo chuỗi mới → nên thao tác trên `list` ký tự rồi `''.join(...)` một lần cuối.

In [2]:
s = "love"
print(s[::-1])
lst = list("abcdef")
lst[1:4] = lst[1:4][::-1]
print(''.join(lst))

evol
adcbef


### 3.2 Stack (ngăn xếp) bằng list

Stack hoạt động kiểu **vào sau ra trước (LIFO)**: `append` để đẩy vào đỉnh, `pop()` để lấy ra từ đỉnh. Ngoặc mở gần nhất chưa đóng luôn nằm ở đỉnh stack → khi gặp `)` ta biết ngay nó khớp với `(` nào.

In [3]:
stack = []
for ch in "abc":
    stack.append(ch)
print(stack, stack.pop(), stack)

['a', 'b'] c ['a', 'b']


### 3.3 Ghép cặp ngoặc bằng stack chỉ số

Duyệt chuỗi; gặp `(` đẩy **chỉ số** vào stack; gặp `)` pop ra chỉ số `j` → cặp `(j, i)`. Lưu `pair[j] = i`, `pair[i] = j`.

In [4]:
def match_pairs(s):
    stack, pair = [], {}
    for i, ch in enumerate(s):
        if ch == '(':
            stack.append(i)
        elif ch == ')':
            j = stack.pop()
            pair[i], pair[j] = j, i
    return pair


print(match_pairs("(u(love)i)"))

{7: 2, 2: 7, 9: 0, 0: 9}


### 3.4 `''.join(list)` thay cho cộng chuỗi trong vòng lặp

Cộng `res += ch` lặp lại có thể tốn `O(n^2)` trong trường hợp xấu (vì tạo chuỗi mới). Gom vào list rồi `''.join` là `O(n)`.

In [5]:
parts = []
for ch in "hello":
    parts.append(ch.upper())
print(''.join(parts))

HELLO


### 3.5 Độ sâu lồng ngoặc

Độ sâu (depth) của một ký tự = số cặp ngoặc bao quanh nó. Một ký tự ở độ sâu `d` bị đảo `d` lần → nếu `d` **lẻ** thì nó nằm ở vị trí "bị lật", nếu `d` **chẵn** thì như cũ (về mặt hướng đọc). Quan sát này dẫn tới mẹo `O(n)`.

In [6]:
def depths(s):
    d, out = 0, []
    for ch in s:
        if ch == '(':
            d += 1
        elif ch == ')':
            d -= 1
        else:
            out.append((ch, d))
    return out


print(depths("(u(love)i)"))

[('u', 1), ('l', 2), ('o', 2), ('v', 2), ('e', 2), ('i', 1)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay `"(ed(et(oc))el)"`:

| Bước | Cặp được đảo | Chuỗi |
|---|---|---|
| 0 | — | `(ed(et(oc))el)` |
| 1 | `(oc)` → `co` | `(ed(etco)el)` |
| 2 | `(etco)` → `octe` | `(edocteel)` |
| 3 | `(edocteel)` → `leetcode` | `leetcode` |

Mỗi bước: tìm một cặp ngoặc **không chứa ngoặc nào bên trong** (cặp trong cùng), đảo phần bên trong, xóa hai ngoặc. Lặp đến khi hết ngoặc.

In [7]:
s = "(ed(et(oc))el)"
while '(' in s:
    r = s.index(')')            # ')' đầu tiên luôn đóng một cặp trong cùng
    l = s.rindex('(', 0, r)     # '(' gần nhất bên trái nó
    s = s[:l] + s[l + 1:r][::-1] + s[r + 1:]
    print(s)

(ed(etco)el)
(edocteel)
leetcode


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đúng như tính tay: lặp lại "tìm `)` đầu tiên, tìm `(` gần nhất bên trái, đảo đoạn giữa, xóa hai ngoặc".

Tại sao `)` đầu tiên luôn thuộc cặp trong cùng? Vì trước nó không có `)` nào, nên giữa nó và `(` khớp với nó không thể có cặp ngoặc hoàn chỉnh nào.

In [8]:
def brute(s: str) -> str:
    while '(' in s:
        r = s.index(')')
        l = s.rindex('(', 0, r)
        s = s[:l] + s[l + 1:r][::-1] + s[r + 1:]
    return s


for s, want in examples:
    print(brute(s), want)
    assert brute(s) == want

dcba dcba
iloveu iloveu
leetcode leetcode
apmnolkjihgfedcbq apmnolkjihgfedcbq


**Độ phức tạp:** có `p` cặp ngoặc (`p ≤ n/2`), mỗi vòng tạo chuỗi mới `O(n)` → `O(n · p) = O(n^2)`. Với `n = 2000` vẫn chạy được (~10^6), nhưng ý tưởng không mở rộng được.

**Nút thắt:** (1) mỗi vòng **quét lại từ đầu** để tìm ngoặc; (2) mỗi ký tự ở độ sâu `d` bị **sao chép/đảo `d` lần**. Trường hợp xấu nhất: `"((((...abc...))))"`.

In [9]:
for depth in [200, 400, 800]:
    s = '(' * depth + 'x' * depth + ')' * depth
    t = time.perf_counter(); brute(s); print(depth, f"{time.perf_counter() - t:.4f}s")

200 0.0001s
400 0.0003s
800 0.0007s


## 6. Bước đột phá

**Insight 1 (stack ngữ cảnh):** khi gặp `(`, phần đã đọc bên ngoài cần "tạm cất" lại; khi gặp `)`, phần bên trong đã hoàn chỉnh (mọi ngoặc con đã được xử lý) → đảo nó rồi **nối vào** ngữ cảnh vừa cất. Không cần quét lại từ đầu.

**Insight 2 (đường hầm `O(n)`):** đảo một đoạn = **đọc nó theo chiều ngược lại**. Thay vì thật sự đảo, ta đi dọc chuỗi với biến hướng `d = +1`. Khi gặp ngoặc tại `i`:
- **Nhảy** sang ngoặc khớp `pair[i]`,
- **Đổi hướng** `d = -d`,
- Tiếp tục bước `i += d`.

Quy tắc: *nếu* đang đứng ở ngoặc *thì* dịch chuyển tức thời sang ngoặc đối diện và quay đầu; *nếu* là chữ cái *thì* ghi ra kết quả. Mỗi vị trí được thăm đúng một lần → `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Stack các đoạn chuỗi (list ký tự)

Giữ `stack` các list. `cur` là list đang xây ở mức hiện tại.
- Chữ cái → `cur.append(ch)`.
- `(` → đẩy `cur` vào stack, mở `cur = []` mới.
- `)` → `cur.reverse()`, lấy `prev = stack.pop()`, `prev.extend(cur)`, `cur = prev`.

In [10]:
def solve_stack_strings(s: str) -> str:
    stack = []
    cur = []
    for ch in s:
        if ch == '(':
            stack.append(cur)
            cur = []
        elif ch == ')':
            cur.reverse()
            prev = stack.pop()
            prev.extend(cur)
            cur = prev
        else:
            cur.append(ch)
    return ''.join(cur)


for s, want in examples:
    assert solve_stack_strings(s) == want
print([solve_stack_strings(s) for s, _ in examples])

['dcba', 'iloveu', 'leetcode', 'apmnolkjihgfedcbq']


**Độ phức tạp:** mỗi `)` đảo + nối đoạn bên trong → ký tự ở độ sâu `d` vẫn bị động chạm `d` lần → worst `O(n^2)`, nhưng chỉ một lượt quét, hằng số nhỏ. Bộ nhớ `O(n)`.

### Cách 3 — Stack chỉ số + đảo tại chỗ trên list kết quả

Duyệt `s`, ghi chữ cái vào `res` (list). Gặp `(` đẩy `len(res)` (vị trí bắt đầu đoạn) vào stack. Gặp `)` pop ra `start` và đảo `res[start:]` tại chỗ. Ngoặc không bao giờ được ghi vào `res`.

In [11]:
def solve_stack_index(s: str) -> str:
    res = []
    stack = []
    for ch in s:
        if ch == '(':
            stack.append(len(res))
        elif ch == ')':
            start = stack.pop()
            res[start:] = res[start:][::-1]
        else:
            res.append(ch)
    return ''.join(res)


for s, want in examples:
    assert solve_stack_index(s) == want
print([solve_stack_index(s) for s, _ in examples])

['dcba', 'iloveu', 'leetcode', 'apmnolkjihgfedcbq']


**Độ phức tạp:** worst `O(n^2)` (giống Cách 2), bộ nhớ `O(n)`. Đây là cách ngắn gọn, phổ biến nhất trong phỏng vấn.

### Cách 4 — Đường hầm (wormhole / teleport) `O(n)`

1. Ghép cặp ngoặc → `pair[i]`.
2. `i = 0`, `d = 1`. Trong khi `0 <= i < n`: nếu `s[i]` là ngoặc thì `i = pair[i]`, `d = -d`; ngược lại ghi `s[i]`. Sau đó `i += d`.

In [12]:
def solve_wormhole(s: str) -> str:
    n = len(s)
    pair = [0] * n
    stack = []
    for i, ch in enumerate(s):
        if ch == '(':
            stack.append(i)
        elif ch == ')':
            j = stack.pop()
            pair[i], pair[j] = j, i
    res = []
    i, d = 0, 1
    while i < n:
        if s[i] in '()':
            i = pair[i]
            d = -d
        else:
            res.append(s[i])
        i += d
    return ''.join(res)


for s, want in examples:
    assert solve_wormhole(s) == want
print([solve_wormhole(s) for s, _ in examples])

['dcba', 'iloveu', 'leetcode', 'apmnolkjihgfedcbq']


**Vì sao vòng lặp dừng?** Mỗi ngoặc được "đi qua" đúng hai lần (một lần mỗi chiều) và mỗi chữ cái được ghi đúng một lần; cuối cùng `i` vượt khỏi `n - 1` theo chiều `+1` (phần ngoài cùng luôn đọc xuôi). **Độ phức tạp:** `O(n)` thời gian và bộ nhớ.

### Cách 5 — Đệ quy theo cấu trúc ngoặc

Hàm `parse(i)` đọc từ vị trí `i` cho tới `)` tương ứng (hoặc hết chuỗi), trả về (list ký tự, vị trí kết thúc). Gặp `(` thì gọi đệ quy, đảo kết quả rồi nối. Độ sâu đệ quy = độ sâu ngoặc ≤ 1000 với `n ≤ 2000` — **có thể chạm giới hạn đệ quy mặc định của Python (1000)**, nên cách này chỉ để minh họa; khi nộp hãy dùng Cách 3 hoặc 4.

In [13]:
def solve_recursive(s: str) -> str:
    def parse(i):
        out = []
        while i < len(s) and s[i] != ')':
            if s[i] == '(':
                inner, i = parse(i + 1)
                out.extend(reversed(inner))
            else:
                out.append(s[i])
            i += 1
        return out, i

    return ''.join(parse(0)[0])


for s, want in examples:
    assert solve_recursive(s) == want
print([solve_recursive(s) for s, _ in examples])

['dcba', 'iloveu', 'leetcode', 'apmnolkjihgfedcbq']


## 8. TRACE từng bước

Trace Cách 4 trên `s = "(u(love)i)"` (chỉ số `0..9`): `pair = {0:9, 2:7, 7:2, 9:0}`.

| bước | i | s[i] | hành động | d sau | res |
|---|---|---|---|---|---|
| 1 | 0 | `(` | nhảy tới 9, đổi chiều | -1 | |
| 2 | 8 | `i` | ghi | -1 | i |
| 3 | 7 | `)` | nhảy tới 2, đổi chiều | +1 | i |
| 4 | 3..6 | `love` | ghi lần lượt | +1 | ilove |
| 5 | 7 | `)` | nhảy tới 2, đổi chiều | -1 | ilove |
| 6 | 1 | `u` | ghi | -1 | iloveu |
| 7 | 0 | `(` | nhảy tới 9, đổi chiều | +1 | iloveu |
| 8 | 10 | — | dừng | | iloveu |

In [14]:
def trace_wormhole(s):
    n = len(s)
    pair, stack = {}, []
    for i, ch in enumerate(s):
        if ch == '(':
            stack.append(i)
        elif ch == ')':
            j = stack.pop(); pair[i], pair[j] = j, i
    print("pair =", pair)
    res, i, d, step = [], 0, 1, 0
    while i < n:
        step += 1
        if s[i] in '()':
            print(f"bước {step:2}: i={i} '{s[i]}' -> nhảy tới {pair[i]}, d: {d} -> {-d}")
            i = pair[i]; d = -d
        else:
            res.append(s[i])
            print(f"bước {step:2}: i={i} '{s[i]}' ghi -> res={''.join(res)}")
        i += d
    return ''.join(res)


trace_wormhole("(u(love)i)")

pair = {7: 2, 2: 7, 9: 0, 0: 9}
bước  1: i=0 '(' -> nhảy tới 9, d: 1 -> -1
bước  2: i=8 'i' ghi -> res=i
bước  3: i=7 ')' -> nhảy tới 2, d: -1 -> 1
bước  4: i=3 'l' ghi -> res=il
bước  5: i=4 'o' ghi -> res=ilo
bước  6: i=5 'v' ghi -> res=ilov
bước  7: i=6 'e' ghi -> res=ilove
bước  8: i=7 ')' -> nhảy tới 2, d: 1 -> -1
bước  9: i=1 'u' ghi -> res=iloveu
bước 10: i=0 '(' -> nhảy tới 9, d: -1 -> 1


'iloveu'

In [15]:
def trace_stack_index(s):
    res, stack = [], []
    for ch in s:
        if ch == '(':
            stack.append(len(res))
        elif ch == ')':
            start = stack.pop()
            res[start:] = res[start:][::-1]
        else:
            res.append(ch)
        print(f"ch={ch}  stack={stack!s:10} res={''.join(res)}")
    return ''.join(res)


trace_stack_index("(ed(et(oc))el)")

ch=(  stack=[0]        res=
ch=e  stack=[0]        res=e
ch=d  stack=[0]        res=ed
ch=(  stack=[0, 2]     res=ed
ch=e  stack=[0, 2]     res=ede
ch=t  stack=[0, 2]     res=edet
ch=(  stack=[0, 2, 4]  res=edet
ch=o  stack=[0, 2, 4]  res=edeto
ch=c  stack=[0, 2, 4]  res=edetoc
ch=)  stack=[0, 2]     res=edetco
ch=)  stack=[0]        res=edocte
ch=e  stack=[0]        res=edoctee
ch=l  stack=[0]        res=edocteel
ch=)  stack=[]         res=leetcode


'leetcode'

## 9. Code nộp LeetCode

Chọn Cách 4 (đường hầm) vì `O(n)`; Cách 3 cũng được chấp nhận với `n ≤ 2000`.

In [16]:
class Solution:
    def reverseParentheses(self, s: str) -> str:
        n = len(s)
        pair = [0] * n
        stack = []
        for i, ch in enumerate(s):
            if ch == '(':
                stack.append(i)
            elif ch == ')':
                j = stack.pop()
                pair[i], pair[j] = j, i
        res = []
        i, d = 0, 1
        while i < n:
            if s[i] == '(' or s[i] == ')':
                i = pair[i]
                d = -d
            else:
                res.append(s[i])
            i += d
        return ''.join(res)


sol = Solution()
for s, want in examples:
    print(sol.reverseParentheses(s))
    assert sol.reverseParentheses(s) == want

dcba
iloveu
leetcode
apmnolkjihgfedcbq


## 10. Vì sao đúng

**Cách 3 (stack chỉ số):** invariant — khi gặp `)`, phần `res[start:]` chính là nội dung bên trong cặp ngoặc này **đã được xử lý xong** (mọi cặp con đã đảo, ngoặc đã bỏ). Đảo nó đúng là thao tác đề yêu cầu cho cặp hiện tại. Vì cặp con luôn đóng trước cặp cha, thứ tự "trong ra ngoài" được đảm bảo.

**Cách 4 (đường hầm):** Xét kết quả cuối cùng như một cách **đọc** chuỗi gốc. Đảo đoạn `(X)` nghĩa là đọc `X` từ phải sang trái. Khi đọc xuôi và gặp `(`, ta phải đọc nội dung từ đầu `)` lùi về → nhảy sang `)` và đi lùi. Khi đang đi lùi bên trong mà gặp `)` của cặp con, nội dung cặp con bị đảo hai lần (một lần bởi cha, một lần bởi chính nó) = đọc xuôi → nhảy sang `(` của nó và đổi chiều thành xuôi. Lập luận quy nạp theo độ sâu: ở độ sâu `d`, hướng đọc là `(-1)^d` — đúng bằng số lần bị đảo. Mỗi ngoặc được đi qua hai lần, mỗi chữ cái một lần → kết quả gồm đủ mọi chữ cái theo đúng thứ tự.

## 11. Time / Space Complexity

Gọi `n = len(s)`, `D` = độ sâu lồng ngoặc lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng lặp | `O(n · p)` ≤ `O(n^2)` | `O(n)` |
| 2. Stack đoạn chuỗi | `O(n · D)` ≤ `O(n^2)` | `O(n)` |
| 3. Stack chỉ số | `O(n · D)` ≤ `O(n^2)` | `O(n)` |
| 4. Đường hầm | `O(n)` | `O(n)` |
| 5. Đệ quy | `O(n · D)` | `O(n)` + ngăn xếp gọi `O(D)` |

Average case (ngoặc nông) thì Cách 2/3 gần tuyến tính; worst case là ngoặc lồng sâu.

## 12. Bộ test

Sinh chuỗi ngoặc cân bằng ngẫu nhiên bằng đệ quy có kiểm soát, đối chiếu mọi cách với brute force.

In [17]:
solvers = {
    "stack_strings": solve_stack_strings,
    "stack_index": solve_stack_index,
    "wormhole": solve_wormhole,
    "recursive": solve_recursive,
    "submit": Solution().reverseParentheses,
}
edge = [
    ("a", "a"),
    ("abc", "abc"),
    ("()", ""),
    ("(())", ""),
    ("a()b", "ab"),
    ("((ab))", "ab"),
    ("(((abc)))", "cba"),
    ("(a)(b)(c)", "abc"),
    ("x(ab)(cd)y", "xbadcy"),
]
for s, want in examples + edge:
    assert brute(s) == want, (s, brute(s))
    for name, f in solvers.items():
        assert f(s) == want, (name, s, f(s), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [18]:
def random_balanced(rng, budget):
    out = []
    while budget > 0:
        r = rng.random()
        if r < 0.35 and budget >= 2:
            inner = rng.randint(0, budget - 2)
            out.append('(' + random_balanced(rng, inner) + ')')
            budget -= inner + 2
        else:
            out.append(rng.choice('abcde'))
            budget -= 1
    return ''.join(out)


rng = random.Random(1190)
for trial in range(1500):
    s = random_balanced(rng, rng.randint(1, 25))
    want = brute(s)
    for name, f in solvers.items():
        assert f(s) == want, (name, s)
print("1500 test ngẫu nhiên khớp brute force; ví dụ:", random_balanced(random.Random(1), 15))

1500 test ngẫu nhiên khớp brute force; ví dụ: (a(db()dac)baa)


In [19]:
deep = '(' * 1000 + ''.join(random.Random(2).choice('xyz') for _ in range(0)) + ')' * 1000   # chỉ ngoặc
assert Solution().reverseParentheses(deep) == ''
deep2 = '(' * 666 + 'a' * 668 + ')' * 666
for name in ["stack_index", "wormhole", "submit"]:
    t = time.perf_counter(); r = solvers[name](deep2); print(f"{name:12s} len={len(r)} {time.perf_counter() - t:.4f}s")
assert solve_stack_index(deep2) == solve_wormhole(deep2) == brute(deep2)
wide = ''.join(random_balanced(random.Random(3), 2000))
assert solve_wormhole(wide) == solve_stack_index(wide)
print("test giới hạn OK")

stack_index  len=668 0.0015s
wormhole     len=668 0.0002s
submit       len=668 0.0002s
test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đảo từ ngoài vào trong (ghép `(` đầu tiên với `)` cuối cùng)

In [20]:
def wrong_outer_first(s):
    while '(' in s:
        l = s.index('(')
        r = s.rindex(')')          # SAI: chưa chắc khớp với '(' này
        s = s[:l] + s[l + 1:r][::-1] + s[r + 1:]
    return s


print(wrong_outer_first("(ab)(cd)"), "đúng là", brute("(ab)(cd)"))

dcba đúng là badc


**Vì sao sai:** `(` đầu tiên và `)` cuối cùng không nhất thiết là một cặp — với `"(ab)(cd)"` chúng thuộc hai cặp khác nhau. Đảo cả cụm `ab)(cd` thành `dc()ba` — ngoặc bị đảo theo, cặp `ab` và `cd` bị trộn lẫn, kết quả cuối `"dcba"` thay vì `"badc"`. **Sửa:** ghép cặp bằng stack, hoặc luôn xử lý `)` đầu tiên với `(` gần nhất bên trái nó.

### Lỗi 2 — Quên bỏ ngoặc khi đảo

In [21]:
def wrong_keep_parens(s):
    res, stack = [], []
    for ch in s:
        if ch == '(':
            stack.append(len(res))
            res.append(ch)            # SAI: ghi cả ngoặc vào kết quả
        elif ch == ')':
            start = stack.pop()
            res[start:] = res[start:][::-1]
            res.append(ch)
        else:
            res.append(ch)
    return ''.join(res)


print(wrong_keep_parens("(u(love)i)"), "đúng là", brute("(u(love)i)"))

i)(loveu() đúng là iloveu


**Vì sao sai:** đề yêu cầu kết quả không chứa ngoặc; hơn nữa ngoặc bị đảo theo làm hỏng vị trí `start` của những lần đảo sau. **Sửa:** không ghi ngoặc vào `res`.

### Lỗi 3 — Đường hầm nhưng quên đổi chiều (hoặc quên nhảy)

In [22]:
def wrong_no_flip(s):
    n = len(s); pair, st = {}, []
    for i, ch in enumerate(s):
        if ch == '(':
            st.append(i)
        elif ch == ')':
            j = st.pop(); pair[i], pair[j] = j, i
    res, i, d = [], 0, 1
    while 0 <= i < n:
        if s[i] in '()':
            d = -d                    # SAI: đổi chiều nhưng không nhảy
        else:
            res.append(s[i])
        i += d
        if len(res) > 3 * n:
            return "VÒNG LẶP VÔ HẠN"
    return ''.join(res)


print(wrong_no_flip("(abcd)"), "| đúng là", brute("(abcd)"))
print(wrong_no_flip("a(bc)d"), "| đúng là", brute("a(bc)d"))

 | đúng là dcba
aa | đúng là acbd


**Vì sao sai:** chỉ đổi chiều tại chỗ khiến ta quay lại đúng những ký tự vừa đọc (hoặc thoát ra ngoài biên trái). Phải **nhảy sang ngoặc đối diện** rồi mới đổi chiều. **Sửa:** `i = pair[i]; d = -d`.

### Lỗi 4 — Cộng chuỗi `res = res + ch` hoặc `res = ch + res` trong vòng lặp
Đúng về kết quả nhưng `ch + res` luôn tạo chuỗi mới dài `|res|` → `O(n^2)` ngay cả với chuỗi không có ngoặc. Dùng list + `join`.

In [23]:
def slow_prepend(s):
    res = ""
    for ch in s:
        res = ch + res
    return res


big = 'a' * 200000
t = time.perf_counter(); slow_prepend(big); t1 = time.perf_counter() - t
t = time.perf_counter(); ''.join(reversed(big)); t2 = time.perf_counter() - t
print(f"prepend: {t1:.3f}s, join: {t2:.4f}s")

prepend: 0.516s, join: 0.0031s


## 14. Mẹo & nhận dạng pattern

- **Ngoặc lồng nhau + "từ trong ra ngoài"** → stack. Cặp đóng trước là cặp trong cùng.
- **Lưu ngữ cảnh khi đi vào cấp mới** (push `cur`, mở `cur` mới) — cùng khuôn với LeetCode 394 (Decode String), 224 (Basic Calculator).
- **Đảo đoạn = đọc ngược** → có thể thay thao tác đảo thật bằng **đổi hướng duyệt**.
- **Tiền xử lý cặp ngoặc** `pair[i]` cho phép "nhảy" `O(1)` — dùng nhiều trong bài xử lý biểu thức, Brainf*ck interpreter.
- Với chuỗi: tránh cộng chuỗi trong vòng lặp; dùng list.

## 15. Biến thể

1. **LeetCode 394 — Decode String:** `3[a2[c]]` → `accaccacc`, cùng kỹ thuật stack ngữ cảnh.
2. **LeetCode 1111 / 1614 — độ sâu ngoặc:** đếm độ sâu bằng biến đếm.
3. **Nhiều loại ngoặc** `()`, `[]`, `{}` với hành vi khác nhau (ví dụ `[]` thì viết hoa) → stack lưu thêm loại ngoặc.
4. **Chỉ đảo các cặp ở độ sâu lẻ / chẵn** → dùng biến depth thay vì đảo thật.
5. **Đề cho ngoặc có thể không cân bằng** → bỏ qua `)` thừa, coi `(` thừa như ký tự thường (cần kiểm tra trước).

In [24]:
def reverse_brackets_upper(s):
    """Biến thể 3: '()' đảo ngược, '[]' viết hoa nội dung."""
    stack, cur = [], []
    for ch in s:
        if ch in '([':
            stack.append((cur, ch)); cur = []
        elif ch in ')]':
            prev, kind = stack.pop()
            inner = cur[::-1] if kind == '(' else [c.upper() for c in cur]
            prev.extend(inner); cur = prev
        else:
            cur.append(ch)
    return ''.join(cur)


print(reverse_brackets_upper("a[bc(de)]f"), reverse_brackets_upper("(ab[cd])"))

aBCEDf DCba


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Tìm `)` đầu tiên, đảo, lặp lại | `O(n^2)` | `O(n)` | Oracle, hiểu đề |
| 2. Stack đoạn chuỗi | push `cur` khi `(`, đảo + nối khi `)` | `O(n·D)` | `O(n)` | Tổng quát cho xử lý cấp lồng |
| 3. Stack chỉ số | đảo `res[start:]` khi `)` | `O(n·D)` | `O(n)` | Phỏng vấn, code ngắn |
| 4. Đường hầm | nhảy sang ngoặc đối diện, đổi chiều | `O(n)` | `O(n)` | Tối ưu, ngoặc lồng sâu |
| 5. Đệ quy | parse theo cấu trúc | `O(n·D)` | `O(n)` + stack gọi | Minh họa, khi ngoặc nông |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `reverseParentheses("ta()usw((((a))))")` = ?

**Bài 2 (tính tay):** trace `res` và `stack` của Cách 3 với `s = "a(bc(de)f)g"`.

**Bài 3 (điền TODO):** hoàn thiện `build_pair(s)` trả về list `pair` (với chữ cái để `-1`).

**Bài 4 (tự viết lại):** viết lại Cách 4 không dùng list `pair` mà dùng dict.

**Bài 5 (mini challenge):** cho biết **mỗi chữ cái** trong kết quả đến từ chỉ số nào của `s` (trả về list chỉ số) — dùng đường hầm.

In [25]:
def build_pair(s):
    # TODO: pair[i] = chỉ số ngoặc khớp với s[i], hoặc -1 nếu s[i] là chữ cái
    pass


def reverse_dict_wormhole(s):
    # TODO
    raise NotImplementedError


def source_indices(s):
    # TODO: trả về list chỉ số gốc của từng ký tự trong kết quả
    raise NotImplementedError

# assert build_pair("a(b)") == [-1, 3, -1, 1]
# assert source_indices("(ab)") == [2, 1]

## 18. Gợi ý 3 mức

- **Mức 1:** Cặp ngoặc nào đóng trước thì là cặp trong cùng — dùng stack.
- **Mức 2:** Khi gặp `)`, nội dung bên trong đã xử lý xong; chỉ cần biết nó bắt đầu ở đâu trong `res`.
- **Mức 3:** Bài 5: trong vòng lặp đường hầm, thay vì `res.append(s[i])` hãy `res.append(i)`. Bài 1: ngoặc rỗng biến mất, cụm `((((a))))` đảo 4 lần vẫn là `a`.

## 19. Đáp án tham khảo

**Bài 1:** `"tauswa"`.

**Bài 2:** `a` → res=`a`; `(` → stack=[1]; `b`,`c` → `abc`; `(` → stack=[1,3]; `d`,`e` → `abcde`; `)` → đảo res[3:] → `abced`, stack=[1]; `f` → `abcedf`; `)` → đảo res[1:] → `afdecb`; `g` → `afdecbg`.

In [26]:
assert Solution().reverseParentheses("ta()usw((((a))))") == "tauswa"
assert solve_stack_index("a(bc(de)f)g") == "afdecbg"


def build_pair(s):
    pair, st = [-1] * len(s), []
    for i, ch in enumerate(s):
        if ch == '(':
            st.append(i)
        elif ch == ')':
            j = st.pop(); pair[i], pair[j] = j, i
    return pair


def reverse_dict_wormhole(s):
    pair, st = {}, []
    for i, ch in enumerate(s):
        if ch == '(':
            st.append(i)
        elif ch == ')':
            j = st.pop(); pair[i], pair[j] = j, i
    res, i, d = [], 0, 1
    while i < len(s):
        if i in pair:
            i, d = pair[i], -d
        else:
            res.append(s[i])
        i += d
    return ''.join(res)


def source_indices(s):
    pair = build_pair(s)
    res, i, d = [], 0, 1
    while i < len(s):
        if pair[i] != -1:
            i, d = pair[i], -d
        else:
            res.append(i)
        i += d
    return res


assert build_pair("a(b)") == [-1, 3, -1, 1]
assert source_indices("(ab)") == [2, 1]
rng = random.Random(19)
for _ in range(800):
    s = random_balanced(rng, rng.randint(1, 20))
    assert reverse_dict_wormhole(s) == brute(s)
    assert ''.join(s[i] for i in source_indices(s)) == brute(s)
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
# Stack chỉ số (ngắn gọn)
res, st = [], []
for ch in s:
    if ch == '(':   st.append(len(res))
    elif ch == ')': k = st.pop(); res[k:] = res[k:][::-1]
    else:           res.append(ch)

# Đường hầm O(n)
pair = ghép cặp bằng stack
i, d = 0, 1
while i < n:
    if s[i] in '()': i, d = pair[i], -d
    else: res.append(s[i])
    i += d
```

**Checklist:**
- [ ] Ngoặc được ghép bằng stack, không phải "`(` đầu với `)` cuối"?
- [ ] Kết quả không còn ngoặc?
- [ ] Ngoặc rỗng `()` và lồng sâu `((((a))))` chạy đúng?
- [ ] Dùng list + `''.join`, không cộng chuỗi trong vòng lặp?
- [ ] Đường hầm: vừa **nhảy** vừa **đổi chiều**?
- [ ] Không dùng đệ quy sâu (độ sâu có thể ~1000)?